# Gated Attention Fusion + XGBoost â€” IEEE-CIS Fraud Detection

**Architecture:** `architecture_justification_v2.md` Â§4.6 (Gated Attention Fusion) + Â§4.7 (XGBoost Meta-Learner)

Combines embeddings from three trained models:
- **TCN** (Keras/TF) â†’ 64-d temporal embedding
- **GAT** (PyTorch/DGL) â†’ 64-d relational embedding
- **CaT-GNN** (PyTorch/DGL) â†’ 64-d causal embedding

Fusion pipeline:
```
TCN(64) + GAT(64) + CaT-GNN(64)
   â†’ GatedFusion â†’ fused(64)
   â†’ [fused(64) || tabular(263)] â†’ XGBoost â†’ fraud probability
```

In [ ]:
!pip install dgl -f https://data.dgl.ai/wheels/torch-2.1/cu121/repo.html -q

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TensorFlow warnings

import numpy as np
import pandas as pd
import pickle, json, gc, time, datetime, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import models
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam
from tcn import TCN

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import dgl
from dgl.nn import GATConv
from dgl.dataloading import MultiLayerFullNeighborSampler
from dgl.dataloading import DataLoader as DGLDataLoader

import matplotlib.pyplot as plt

# GPU Configuration: Keep GPU enabled for PyTorch
# TensorFlow TCN will use CPU explicitly in inference (no global disable)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device (PyTorch): {DEVICE}')
print(f'TF: {tf.__version__} | PyTorch: {torch.__version__} | DGL: {dgl.__version__}')
print('Note: TCN inference will run on CPU, PyTorch/XGBoost use GPU')

np.random.seed(42)
tf.random.set_seed(42)
torch.manual_seed(42)

In [ ]:
# â”€â”€ Paths â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
DATA_DIR            = '/kaggle/input/ieee-fraud-detection'
TCN_WEIGHTS_PATH    = '/kaggle/input/tcn-weights/tcn_fraud_best_model.weights.h5'
TCN_SCALER_PATH     = '/kaggle/input/tcn-weights/tcn_scaler.pkl'
GAT_WEIGHTS_PATH    = 'gat_results/models/gat_best.pt'
CATGNN_WEIGHTS_PATH = 'catgnn_results/models/catgnn_best.pt'
SAVE_DIR            = 'fusion_results'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.join(SAVE_DIR, 'models'), exist_ok=True)

# â”€â”€ Graph config (GTAN methodology) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
GRAPH_K_NEIGHBORS = 3
MAX_GROUP_SIZE    = 5000

# â”€â”€ TCN config (must match experiment_tcn exactly) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
SEQUENCE_LENGTH = 20
TCN_FILTERS     = 128
KERNEL_SIZE     = 3
DILATIONS       = [1, 2, 4, 8]
DROPOUT_RATE    = 0.2
DENSE_UNITS     = [128, 64, 32]

# â”€â”€ Model dims â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
HIDDEN_DIM = 64
NUM_HEADS  = 4
FEAT_DROP  = 0.3
ATTN_DROP  = 0.3

# â”€â”€ Fusion training â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
FUSION_LR     = 1e-3
FUSION_EPOCHS = 50
FUSION_BATCH  = 1024

print('Configuration loaded.')

In [ ]:
%%time
X_train  = pd.read_csv(f'{DATA_DIR}/train_transaction.csv')
X_test   = pd.read_csv(f'{DATA_DIR}/test_transaction.csv')
train_id = pd.read_csv(f'{DATA_DIR}/train_identity.csv')
test_id  = pd.read_csv(f'{DATA_DIR}/test_identity.csv')

for df in [train_id, test_id, X_train, X_test]:
    df.columns = [c.replace('-', '_') if c.startswith('id') else c for c in df.columns]

X_train = X_train.merge(train_id, on='TransactionID', how='left')
X_test  = X_test.merge(test_id,  on='TransactionID', how='left')

y_train = X_train['isFraud'].copy()
del X_train['isFraud']
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)

print(f'Train: {X_train.shape},  Test: {X_test.shape}')
print(f'Fraud rate: {y_train.mean()*100:.2f}%')

In [ ]:
V_COLS  = [1,3,4,6,8,11,13,14,17,20,23,26,27,30]
V_COLS += [36,37,40,41,44,47,48,54,56,59,62,65,67,68,70]
V_COLS += [76,78,80,82,86,88,89,91]
V_COLS += [107,108,111,115,117,120,121,123,124,127,129,130,136]
V_COLS += [138,139,142,147,156,162,165,160,166]
V_COLS += [178,176,173,182,187,203,205,207,215]
V_COLS += [169,171,175,180,185,188,198,210,209]
V_COLS += [218,223,224,226,228,229,235,220,221,234,238,250,271]
V_COLS += [240,258,257,253,252,260,261,264,266,267,274,277]
V_COLS += [294,284,285,286,291,297,303,305,307,309,310,320]
V_COLS += [281,283,289,296,301,314]

CAT_COLS = [
    'ProductCD','card4','card6','P_emaildomain','R_emaildomain',
    'M1','M2','M3','M4','M5','M6','M7','M8','M9',
    'id_12','id_15','id_16','id_23','id_27','id_28','id_29',
    'id_30','id_31','id_33','id_34','id_35','id_36','id_37','id_38',
    'DeviceType','DeviceInfo'
]

v_keep = {f'V{i}' for i in V_COLS}
v_drop = [c for c in X_train.columns if c.startswith('V') and c not in v_keep]
X_train.drop(columns=v_drop, inplace=True)
X_test.drop(columns=v_drop, inplace=True)

for col in CAT_COLS:
    if col in X_train.columns: X_train[col] = X_train[col].astype('category')
    if col in X_test.columns:  X_test[col]  = X_test[col].astype('category')

print(f'Shape after V-col drop: {X_train.shape}')

In [ ]:
# Sort by TransactionDT (required for GTAN temporal graph)
X_train['__label__'] = y_train.values
X_train = X_train.sort_values('TransactionDT').reset_index(drop=True)
y_train = X_train.pop('__label__').astype('int8')
X_test  = X_test.sort_values('TransactionDT').reset_index(drop=True)

X_train['day'] = X_train['TransactionDT'] / np.float32(24*60*60)
X_test['day']  = X_test['TransactionDT']  / np.float32(24*60*60)

D_SKIP = {1, 2, 3, 5, 9}
for i in range(1, 16):
    col = f'D{i}'
    if i in D_SKIP: continue
    if col in X_train.columns: X_train[col] = X_train[col] - X_train['day']
    if col in X_test.columns:  X_test[col]  = X_test[col]  - X_test['day']

START_DATE = datetime.datetime.strptime('2017-11-30', '%Y-%m-%d')
for df in [X_train, X_test]:
    dt_s = df['TransactionDT'].apply(lambda x: START_DATE + datetime.timedelta(seconds=x))
    df['DT_M'] = (dt_s.dt.year - 2017)*12 + dt_s.dt.month

print('Sort + D-transform + DT_M done.')

In [ ]:
%%time
SKIP_ENCODE = {'TransactionAmt','TransactionDT','day','DT_M'}
shared_cols = set(X_train.columns) & set(X_test.columns)

for f in list(X_train.columns):
    if (str(X_train[f].dtype)=='category') or (X_train[f].dtype=='object'):
        if f in shared_cols:
            df_comb = pd.concat([X_train[f], X_test[f]], axis=0)
            df_comb, _ = df_comb.factorize(sort=True)
            X_train[f] = df_comb[:len(X_train)].astype('int16')
            X_test[f]  = df_comb[len(X_train):].astype('int16')
    elif f not in SKIP_ENCODE:
        mn = np.min((X_train[f].min(), X_test[f].min()))
        X_train[f] = X_train[f] - np.float32(mn)
        X_test[f]  = X_test[f]  - np.float32(mn)
        X_train[f].fillna(-1, inplace=True)
        X_test[f].fillna(-1, inplace=True)

print('Label encoding + normalization done.')

In [ ]:
def encode_FE(df1, df2, cols):
    for col in cols:
        df = pd.concat([df1[col], df2[col]])
        vc = df.value_counts(dropna=True, normalize=True).to_dict()
        vc[-1] = -1
        nm = col+'_FE'
        df1[nm] = df1[col].map(vc).astype('float32')
        df2[nm] = df2[col].map(vc).astype('float32')
        print(nm, ', ', end='')

def encode_LE(col, train, test, verbose=True):
    df_comb = pd.concat([train[col], test[col]], axis=0)
    df_comb, _ = df_comb.factorize(sort=True)
    t = 'int32' if df_comb.max()>32000 else 'int16'
    train[col] = df_comb[:len(train)].astype(t)
    test[col]  = df_comb[len(train):].astype(t)
    del df_comb; gc.collect()
    if verbose: print(col, ', ', end='')

def encode_CB(col1, col2, df1=None, df2=None):
    if df1 is None: df1, df2 = X_train, X_test
    nm = col1+'_'+col2
    df1[nm] = df1[col1].astype(str)+'_'+df1[col2].astype(str)
    df2[nm] = df2[col1].astype(str)+'_'+df2[col2].astype(str)
    encode_LE(nm, df1, df2, verbose=False)
    print(nm, ', ', end='')

def encode_AG(main_columns, uids, aggregations=['mean'], train_df=None, test_df=None, fillna=True, usena=False):
    if train_df is None: train_df, test_df = X_train, X_test
    for main_column in main_columns:
        for col in uids:
            for agg_type in aggregations:
                nm = main_column+'_'+col+'_'+agg_type
                tmp = pd.concat([train_df[[col,main_column]], test_df[[col,main_column]]])
                if usena: tmp.loc[tmp[main_column]==-1, main_column] = np.nan
                tmp = tmp.groupby([col])[main_column].agg([agg_type]).reset_index()
                tmp.rename(columns={agg_type: nm}, inplace=True)
                tmp.index = list(tmp[col])
                tmp = tmp[nm].to_dict()
                train_df[nm] = train_df[col].map(tmp).astype('float32')
                test_df[nm]  = test_df[col].map(tmp).astype('float32')
                if fillna:
                    train_df[nm].fillna(-1, inplace=True)
                    test_df[nm].fillna(-1, inplace=True)
                print("'"+nm+"'", ', ', end='')

def encode_AG2(main_columns, uids, train_df=None, test_df=None):
    if train_df is None: train_df, test_df = X_train, X_test
    for main_column in main_columns:
        for col in uids:
            comb = pd.concat([train_df[[col,main_column]], test_df[[col,main_column]]], axis=0)
            mp = comb.groupby(col)[main_column].agg(['nunique'])['nunique'].to_dict()
            nm = col+'_'+main_column+'_ct'
            train_df[nm] = train_df[col].map(mp).astype('float32')
            test_df[nm]  = test_df[col].map(mp).astype('float32')
            print(nm+', ', end='')

print('Encoding functions defined.')

In [ ]:
%%time
X_train['cents'] = (X_train['TransactionAmt']-np.floor(X_train['TransactionAmt'])).astype('float32')
X_test['cents']  = (X_test['TransactionAmt'] -np.floor(X_test['TransactionAmt'])).astype('float32')
print('cents, ', end='')
encode_FE(X_train, X_test, ['addr1','card1','card2','card3','P_emaildomain'])
encode_CB('card1','addr1')
encode_CB('card1_addr1','P_emaildomain')
encode_FE(X_train, X_test, ['card1_addr1','card1_addr1_P_emaildomain'])
encode_AG(['TransactionAmt','D9','D11'],['card1','card1_addr1','card1_addr1_P_emaildomain'],['mean','std'],usena=True)
print('\nBasic FE done.')

In [ ]:
X_train['uid'] = (X_train['card1_addr1'].astype(str)+'_'+
                  np.floor(X_train['day']-X_train['D1']).astype(str))
X_test['uid']  = (X_test['card1_addr1'].astype(str)+'_'+
                  np.floor(X_test['day'] -X_test['D1']).astype(str))

# Save raw relation cols for graph (before further encoding)
uid_train    = X_train['uid'].copy()
card1_train  = X_train['card1'].copy()
pemail_train = X_train['P_emaildomain'].copy()
device_train = X_train['DeviceInfo'].copy()

print(f'Unique UIDs: {uid_train.nunique():,}')

In [ ]:
%%time
encode_FE(X_train, X_test, ['uid'])
encode_AG(['TransactionAmt','D4','D9','D10','D15'],['uid'],['mean','std'],fillna=True,usena=True)
encode_AG(['C'+str(x) for x in range(1,15) if x!=3],['uid'],['mean'],fillna=True,usena=True)
encode_AG(['M'+str(x) for x in range(1,10)],['uid'],['mean'],fillna=True,usena=True)
encode_AG(['C14'],['uid'],['std'],fillna=True,usena=True)
encode_AG2(['P_emaildomain','dist1','DT_M','id_02','cents'],['uid'])
encode_AG2(['C13','V314'],['uid'])
encode_AG2(['V127','V136','V309','V307','V320'],['uid'])
X_train['outsider15'] = (np.abs(X_train['D1']-X_train['D15'])>3).astype('int8')
X_test['outsider15']  = (np.abs(X_test['D1'] -X_test['D15'])>3).astype('int8')
print('\nAggregation + outsider15 done.')

In [ ]:
DROP_COLS = (
    ['TransactionDT','TransactionID'] +
    ['D6','D7','D8','D9','D12','D13','D14'] +
    ['DT_M','day','uid'] +
    ['C3','M5','id_08','id_33'] +
    ['card4','id_07','id_14','id_21','id_30','id_32','id_34'] +
    ['id_'+str(x) for x in range(22,28)]
)
cols = [c for c in X_train.columns if c not in DROP_COLS]

X_all = X_train[cols].copy().fillna(-1).astype('float32').values  # (N, n_features)
y_all = y_train.values.astype('int64')

IN_DIM = X_all.shape[1]
print(f'Node feature matrix: {X_all.shape}  IN_DIM={IN_DIM}')
print(f'Fraud rate: {y_all.mean()*100:.2f}%')

In [ ]:
# Stratified split on node IDs (transductive â€” all nodes in one graph)
all_nids_np = np.arange(len(y_all))
train_nids_np, val_nids_np = train_test_split(
    all_nids_np, test_size=0.2, stratify=y_all, random_state=42
)
train_nids = torch.LongTensor(train_nids_np)
val_nids   = torch.LongTensor(val_nids_np)

y_train_split = y_all[train_nids_np]
y_val_split   = y_all[val_nids_np]

print(f'Train: {len(train_nids):,}  Val: {len(val_nids):,}')
print(f'Train fraud: {y_train_split.mean()*100:.2f}%  Val fraud: {y_val_split.mean()*100:.2f}%')

# Scale node features (fit on train nodes only)
graph_scaler = StandardScaler()
X_scaled = X_all.copy()
X_scaled[train_nids_np] = graph_scaler.fit_transform(X_all[train_nids_np])
X_scaled[val_nids_np]   = graph_scaler.transform(X_all[val_nids_np])

node_feat   = torch.FloatTensor(X_scaled)
node_labels = torch.LongTensor(y_all)
print(f'node_feat: {node_feat.shape}')

In [ ]:
def build_gtan_graph(n_nodes, relation_series_dict, k=3, max_group=5000):
    src_list, dst_list = [], []
    stats = {}
    for rel_name, series in relation_series_dict.items():
        n_before = len(src_list)
        skipped_large = 0
        for group_val, group_idx in series.groupby(series).groups.items():
            if pd.isna(group_val) or group_val in (-1, '-1', 'nan'): continue
            indices = list(group_idx)
            n = len(indices)
            if n < 2: continue
            if n > max_group: skipped_large += 1; continue
            for i in range(1, n):
                dst = indices[i]
                for src in indices[max(0,i-k):i]:
                    src_list.append(src)
                    dst_list.append(dst)
        stats[rel_name] = {'edges': len(src_list)-n_before, 'skipped_large': skipped_large}
        print(f'  [{rel_name}] edges={stats[rel_name]["edges"]:,}  skipped_large={skipped_large}')
    g = dgl.graph((np.array(src_list,dtype=np.int64), np.array(dst_list,dtype=np.int64)), num_nodes=n_nodes)
    g = dgl.add_reverse_edges(g)
    g = dgl.add_self_loop(g)
    print(f'Graph: {g.num_nodes():,} nodes, {g.num_edges():,} edges, avg_deg={g.num_edges()/g.num_nodes():.1f}')
    return g, stats

print(f'Building GTAN graph on {len(y_all):,} transactions...')
t0 = time.time()
g, graph_stats = build_gtan_graph(
    n_nodes=len(y_all),
    relation_series_dict={
        'uid':           uid_train.reset_index(drop=True),
        'card1':         card1_train.reset_index(drop=True),
        'P_emaildomain': pemail_train.reset_index(drop=True),
        'DeviceInfo':    device_train.reset_index(drop=True),
    },
    k=GRAPH_K_NEIGHBORS, max_group=MAX_GROUP_SIZE
)
g.ndata['feat']  = node_feat
g.ndata['label'] = node_labels
print(f'Graph built in {time.time()-t0:.1f}s')

## 1. TCN Embedding Extraction

Load TCN weights, extract 64-d embeddings from `dense_2` layer (DENSE_UNITS=[128,64,32]).

In [ ]:
def build_tcn_model(input_shape):
    inputs  = Input(shape=input_shape, name='input_layer')
    tcn_out = TCN(
        nb_filters=TCN_FILTERS, kernel_size=KERNEL_SIZE, nb_stacks=1,
        dilations=DILATIONS, padding='causal', use_skip_connections=True,
        dropout_rate=DROPOUT_RATE, return_sequences=False, activation='relu',
        use_batch_norm=True, use_layer_norm=False, name='tcn_layer'
    )(inputs)
    x = tcn_out
    for i, units in enumerate(DENSE_UNITS):
        x = Dense(units, activation='relu', name=f'dense_{i+1}')(x)
        x = BatchNormalization(name=f'bn_{i+1}')(x)
        x = Dropout(DROPOUT_RATE, name=f'dropout_{i+1}')(x)
    outputs = Dense(1, activation='sigmoid', name='output')(x)
    return models.Model(inputs=inputs, outputs=outputs, name='TCN_FraudDetection')

n_features = IN_DIM
remainder  = n_features % SEQUENCE_LENGTH
pad_width  = (SEQUENCE_LENGTH - remainder) % SEQUENCE_LENGTH
features_per_timestep = (n_features + pad_width) // SEQUENCE_LENGTH
print(f'TCN input: ({SEQUENCE_LENGTH}, {features_per_timestep}), pad_width={pad_width}')

tcn_model = build_tcn_model(input_shape=(SEQUENCE_LENGTH, features_per_timestep))

# Verify model structure
tcn_model.summary()

tcn_model.load_weights(TCN_WEIGHTS_PATH)
print('TCN weights loaded.')

tcn_embedder = keras.Model(
    inputs=tcn_model.input,
    outputs=tcn_model.get_layer('dense_2').output  # 64-d embedding
)
print(f'TCN embedder output: {tcn_embedder.output_shape}')

In [ ]:
with open(TCN_SCALER_PATH, 'rb') as f:
    tcn_scaler = pickle.load(f)
print('TCN scaler loaded.')

X_tcn_scaled = tcn_scaler.transform(X_all)  # (N, n_features)

if pad_width > 0:
    X_tcn_padded = np.hstack([X_tcn_scaled,
                              np.zeros((len(X_tcn_scaled), pad_width), dtype='float32')])
else:
    X_tcn_padded = X_tcn_scaled

X_tcn_3d = X_tcn_padded.reshape(-1, SEQUENCE_LENGTH, features_per_timestep).astype('float32')
print(f'TCN input shape: {X_tcn_3d.shape}')

tcn_emb_all = tcn_embedder.predict(X_tcn_3d, batch_size=512, verbose=1)  # (N, 64)
tcn_emb_train = tcn_emb_all[train_nids_np]
tcn_emb_val   = tcn_emb_all[val_nids_np]
print(f'TCN train: {tcn_emb_train.shape}  val: {tcn_emb_val.shape}')
del X_tcn_scaled, X_tcn_padded, X_tcn_3d; gc.collect()

## 2. GAT Embedding Extraction

In [ ]:
class FraudGAT(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, num_heads=4, feat_drop=0.3, attn_drop=0.3):
        super().__init__()
        self.gat1 = GATConv(in_dim, hidden_dim, num_heads, feat_drop, attn_drop,
                             activation=F.elu, residual=False, allow_zero_in_degree=True)
        self.bn1  = nn.BatchNorm1d(hidden_dim * num_heads)
        self.gat2 = GATConv(hidden_dim*num_heads, hidden_dim, 1, feat_drop, attn_drop,
                             activation=None, residual=True, allow_zero_in_degree=True)
        self.bn2  = nn.BatchNorm1d(hidden_dim)
        self.drop = nn.Dropout(feat_drop)

    def forward(self, blocks, x):
        h = self.gat1(blocks[0], x).flatten(1)
        h = self.drop(self.bn1(h))
        h = self.gat2(blocks[1], h).squeeze(1)
        return self.bn2(h)

gat_model = FraudGAT(in_dim=IN_DIM, hidden_dim=HIDDEN_DIM, num_heads=NUM_HEADS,
                     feat_drop=FEAT_DROP, attn_drop=ATTN_DROP)
gat_model.load_state_dict(torch.load(GAT_WEIGHTS_PATH, map_location='cpu'))
gat_model.eval()
print('GAT model loaded.')

all_nids_t  = torch.arange(g.num_nodes())
loader_gat  = DGLDataLoader(g, all_nids_t, MultiLayerFullNeighborSampler(2),
                             batch_size=4096, shuffle=False, drop_last=False)
gat_emb_list = []
with torch.no_grad():
    for input_nodes, output_nodes, blocks in loader_gat:
        h = gat_model(blocks, node_feat[input_nodes])
        gat_emb_list.append(h.cpu().numpy())

gat_emb_all   = np.vstack(gat_emb_list)
gat_emb_train = gat_emb_all[train_nids_np]
gat_emb_val   = gat_emb_all[val_nids_np]
print(f'GAT train: {gat_emb_train.shape}  val: {gat_emb_val.shape}')

## 3. CaT-GNN Embedding Extraction

In [ ]:
class CausalInspector(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, num_heads=4, feat_drop=0.3, attn_drop=0.3):
        super().__init__()
        embed_dim = hidden_dim * 2
        self.feat_embed = nn.Sequential(
            nn.Linear(in_dim, embed_dim), nn.LayerNorm(embed_dim), nn.ReLU(), nn.Dropout(feat_drop))
        self.attn_layer = GATConv(embed_dim, hidden_dim, num_heads, feat_drop, attn_drop,
                                   activation=F.elu, residual=False, allow_zero_in_degree=True)
        self.causal_proj = nn.Sequential(nn.Linear(embed_dim, hidden_dim), nn.ReLU())
        self.env_proj    = nn.Sequential(nn.Linear(embed_dim, hidden_dim), nn.ReLU())

    def forward(self, g, h):
        h_embed = self.feat_embed(h)
        if g.is_block:
            n_dst = g.num_dst_nodes()
            _, attn = self.attn_layer(g, (h_embed, h_embed[:n_dst]), get_attention=True)
            h_src = h_embed
        else:
            n_dst = g.num_nodes()
            _, attn = self.attn_layer(g, h_embed, get_attention=True)
            h_src = h_embed
        edge_imp = attn.mean(dim=1).squeeze(-1)
        src_idx, dst_idx = g.edges()
        dst_sum = torch.zeros(n_dst, device=edge_imp.device)
        dst_cnt = torch.zeros(n_dst, device=edge_imp.device)
        dst_sum.scatter_add_(0, dst_idx, edge_imp)
        dst_cnt.scatter_add_(0, dst_idx, torch.ones_like(edge_imp))
        thr = (dst_sum/(dst_cnt+1e-8))[dst_idx]
        c_mask = (edge_imp >= thr).float()
        e_mask = 1.0 - c_mask
        c_w = edge_imp * c_mask
        e_w = (thr - edge_imp).clamp(min=0) * e_mask
        D = h_src.shape[-1]
        de = dst_idx.unsqueeze(-1).expand(-1, D)
        hcs = torch.zeros(n_dst, D, device=h_src.device)
        hes = torch.zeros(n_dst, D, device=h_src.device)
        cws = torch.zeros(n_dst, device=h_src.device)
        ews = torch.zeros(n_dst, device=h_src.device)
        hcs.scatter_add_(0, de, h_src[src_idx]*c_w.unsqueeze(-1))
        hes.scatter_add_(0, de, h_src[src_idx]*e_w.unsqueeze(-1))
        cws.scatter_add_(0, dst_idx, c_w)
        ews.scatter_add_(0, dst_idx, e_w)
        hca = hcs/(cws.unsqueeze(-1)+1e-8)
        hea = hes/(ews.unsqueeze(-1)+1e-8)
        nc = (cws==0).unsqueeze(-1); ne = (ews==0).unsqueeze(-1)
        if nc.any() or ne.any():
            ha=torch.zeros(n_dst,D,device=h_src.device); ac=torch.zeros(n_dst,device=h_src.device)
            ha.scatter_add_(0,de,h_src[src_idx]); ac.scatter_add_(0,dst_idx,torch.ones_like(c_w))
            ha=ha/(ac.unsqueeze(-1)+1e-8)
            hca=torch.where(nc,ha,hca); hea=torch.where(ne,ha,hea)
        return self.causal_proj(hca), self.env_proj(hea)


class CausalIntervener(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.blend_net = nn.Sequential(
            nn.Linear(hidden_dim*2, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 2))
    def forward(self, h_causal, h_env):
        if self.training:
            w = F.softmax(self.blend_net(torch.cat([h_causal, h_env], dim=-1)), dim=-1)
            return w[:,0:1]*h_causal + w[:,1:2]*h_env
        return h_env


class CausalTemporalGNN(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, num_heads=4, feat_drop=0.3, attn_drop=0.3):
        super().__init__()
        self.inspector  = CausalInspector(in_dim, hidden_dim, num_heads, feat_drop, attn_drop)
        self.intervener = CausalIntervener(hidden_dim)
        self.ln_causal  = nn.LayerNorm(hidden_dim)
        self.ln_env     = nn.LayerNorm(hidden_dim)
        self.combine    = nn.Linear(hidden_dim*2, hidden_dim)
        self.norm       = nn.LayerNorm(hidden_dim)
        self.output_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(feat_drop))
    def forward(self, g, x):
        hc, he = self.inspector(g, x)
        hc, he = self.ln_causal(hc), self.ln_env(he)
        hei = self.intervener(hc, he)
        h = F.relu(self.combine(torch.cat([hc, hei], dim=-1)))
        return self.output_proj(self.norm(h))

catgnn_model = CausalTemporalGNN(in_dim=IN_DIM, hidden_dim=HIDDEN_DIM,
                                  num_heads=NUM_HEADS, feat_drop=FEAT_DROP, attn_drop=ATTN_DROP)
catgnn_model.load_state_dict(torch.load(CATGNN_WEIGHTS_PATH, map_location='cpu'))
catgnn_model.eval()
print('CaT-GNN model loaded.')

loader_cat = DGLDataLoader(g, all_nids_t, MultiLayerFullNeighborSampler(1),
                            batch_size=2048, shuffle=False, drop_last=False)
cat_emb_list = []
with torch.no_grad():
    for input_nodes, output_nodes, blocks in loader_cat:
        h = catgnn_model(blocks[0], node_feat[input_nodes])
        cat_emb_list.append(h.cpu().numpy())

catgnn_emb_all   = np.vstack(cat_emb_list)
catgnn_emb_train = catgnn_emb_all[train_nids_np]
catgnn_emb_val   = catgnn_emb_all[val_nids_np]
print(f'CaT-GNN train: {catgnn_emb_train.shape}  val: {catgnn_emb_val.shape}')

In [ ]:
# Sanity check: all embeddings aligned, no NaN/Inf
for name, etr, evl in [('TCN',tcn_emb_train,tcn_emb_val),
                        ('GAT',gat_emb_train,gat_emb_val),
                        ('CaT-GNN',catgnn_emb_train,catgnn_emb_val)]:
    assert etr.shape==(len(train_nids),64), f'{name} train shape: {etr.shape}'
    assert evl.shape==(len(val_nids),64),   f'{name} val shape: {evl.shape}'
    assert np.isfinite(etr).all() and np.isfinite(evl).all(), f'{name} has NaN/Inf'
    print(f'{name:8s} train={etr.shape} val={evl.shape} mean={etr.mean():.3f} std={etr.std():.3f}')
print('All embeddings aligned!')

## 4. Gated Attention Fusion (arch_justification_v2 Â§4.6)

```
TCN(64) â†’ Linear(64â†’128) â†’ ReLU â†’ Dropout(0.3)
GAT(64) â†’ Linear(64â†’128) â†’ ReLU â†’ Dropout(0.3)
CaT(64) â†’ Linear(64â†’128) â†’ ReLU â†’ Dropout(0.3)
   Concat(384) â†’ Linear(384â†’128) â†’ ReLU â†’ Dropout â†’ Linear(128â†’3) â†’ Softmax
   â†’ w_tcn, w_gat, w_cat (per-sample, sum=1)
   â†’ Weighted sum(128) â†’ Linear(128â†’64) â†’ ReLU â†’ Dropout â†’ LayerNorm(64)
```

In [ ]:
class GatedFusion(nn.Module):
    """Gated Attention Fusion â€” arch_justification_v2 Â§4.6"""
    def __init__(self, embed_dim=64, proj_dim=128, dropout=0.3):
        super().__init__()
        self.proj_tcn    = nn.Sequential(nn.Linear(embed_dim, proj_dim), nn.ReLU(), nn.Dropout(dropout))
        self.proj_gat    = nn.Sequential(nn.Linear(embed_dim, proj_dim), nn.ReLU(), nn.Dropout(dropout))
        self.proj_catgnn = nn.Sequential(nn.Linear(embed_dim, proj_dim), nn.ReLU(), nn.Dropout(dropout))
        self.gate = nn.Sequential(
            nn.Linear(proj_dim*3, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, 3))
        self.out_proj = nn.Sequential(
            nn.Linear(proj_dim, embed_dim), nn.ReLU(), nn.Dropout(dropout))
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, h_tcn, h_gat, h_catgnn):
        pt = self.proj_tcn(h_tcn)       # (B,128)
        pg = self.proj_gat(h_gat)       # (B,128)
        pc = self.proj_catgnn(h_catgnn) # (B,128)
        w  = F.softmax(self.gate(torch.cat([pt,pg,pc], dim=-1)), dim=-1)  # (B,3)
        fused = w[:,0:1]*pt + w[:,1:2]*pg + w[:,2:3]*pc  # (B,128)
        return self.norm(self.out_proj(fused)), w          # (B,64), (B,3)


class FusionClassifier(nn.Module):
    def __init__(self, embed_dim=64):
        super().__init__()
        self.fusion = GatedFusion(embed_dim=embed_dim)
        self.head   = nn.Sequential(
            nn.Linear(embed_dim, 32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32, 1))
    def forward(self, h_tcn, h_gat, h_cat):
        fused, w = self.fusion(h_tcn, h_gat, h_cat)
        return self.head(fused), w


class FocalLoss(nn.Module):
    """Focal Loss (Lin et al. 2017)"""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt  = torch.exp(-bce)
        return (self.alpha*(1-pt)**self.gamma*bce).mean()

print('Fusion model classes defined.')

## 5. Train Gated Attention Fusion

In [ ]:
t_tcn_tr  = torch.FloatTensor(tcn_emb_train)
t_gat_tr  = torch.FloatTensor(gat_emb_train)
t_cat_tr  = torch.FloatTensor(catgnn_emb_train)
t_y_tr    = torch.FloatTensor(y_train_split)

t_tcn_val = torch.FloatTensor(tcn_emb_val)
t_gat_val = torch.FloatTensor(gat_emb_val)
t_cat_val = torch.FloatTensor(catgnn_emb_val)

fusion_clf = FusionClassifier().to(DEVICE)
optimizer  = torch.optim.Adam(fusion_clf.parameters(), lr=FUSION_LR, weight_decay=1e-4)
criterion  = FocalLoss(alpha=0.25, gamma=2.0)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, min_lr=1e-7)

dataset = TensorDataset(t_tcn_tr, t_gat_tr, t_cat_tr, t_y_tr)
loader  = DataLoader(dataset, batch_size=FUSION_BATCH, shuffle=True)

best_auc, best_state, history = 0.0, None, []

for epoch in range(FUSION_EPOCHS):
    fusion_clf.train()
    epoch_loss = 0.0
    for h1, h2, h3, yb in loader:
        h1,h2,h3,yb = h1.to(DEVICE),h2.to(DEVICE),h3.to(DEVICE),yb.to(DEVICE)
        optimizer.zero_grad()
        logits, _ = fusion_clf(h1, h2, h3)
        loss = criterion(logits.squeeze(-1), yb)
        loss.backward(); optimizer.step()
        epoch_loss += loss.item()

    fusion_clf.eval()
    with torch.no_grad():
        vl, vw = fusion_clf(t_tcn_val.to(DEVICE), t_gat_val.to(DEVICE), t_cat_val.to(DEVICE))
        vp = torch.sigmoid(vl.squeeze(-1)).cpu().numpy()
    auc = roc_auc_score(y_val_split, vp)
    scheduler.step(1-auc)
    history.append({'epoch': epoch+1, 'loss': epoch_loss/len(loader), 'val_auc': auc})
    if auc > best_auc:
        best_auc  = auc
        best_state = {k: v.clone() for k,v in fusion_clf.state_dict().items()}
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{FUSION_EPOCHS} | loss={epoch_loss/len(loader):.4f} | '
              f'val_AUC={auc:.4f} | lr={optimizer.param_groups[0]["lr"]:.2e}')

fusion_clf.load_state_dict(best_state)
torch.save(fusion_clf.state_dict(), os.path.join(SAVE_DIR,'models','fusion_best.pt'))
print(f'Best Fusion AUC: {best_auc:.4f}')

In [ ]:
fusion_clf.eval()
fusion_clf.to(DEVICE)
with torch.no_grad():
    fused_train, gw_train = fusion_clf.fusion(t_tcn_tr.to(DEVICE), t_gat_tr.to(DEVICE), t_cat_tr.to(DEVICE))
    fused_val,   gw_val   = fusion_clf.fusion(t_tcn_val.to(DEVICE), t_gat_val.to(DEVICE), t_cat_val.to(DEVICE))

fused_train = fused_train.cpu().numpy()
fused_val   = fused_val.cpu().numpy()
gw_train    = gw_train.cpu().numpy()
gw_val      = gw_val.cpu().numpy()

assert abs(gw_train.sum(axis=1).mean()-1.0) < 1e-5

print(f'Fused: train={fused_train.shape}  val={fused_val.shape}')
avg = gw_train.mean(axis=0)
print(f'Mean gate weights: TCN={avg[0]:.3f}  GAT={avg[1]:.3f}  CaT-GNN={avg[2]:.3f}')

fraud_tr = y_train_split==1; benign_tr = y_train_split==0
print(f'Fraud:  TCN={gw_train[fraud_tr,0].mean():.3f} GAT={gw_train[fraud_tr,1].mean():.3f} CaT={gw_train[fraud_tr,2].mean():.3f}')
print(f'Benign: TCN={gw_train[benign_tr,0].mean():.3f} GAT={gw_train[benign_tr,1].mean():.3f} CaT={gw_train[benign_tr,2].mean():.3f}')

## 6. XGBoost Meta-Learner (arch_justification_v2 Â§4.7)

Input: fused(64) + tabular(263) = 327 features.

In [ ]:
xgb_scaler  = StandardScaler()
X_tr_tab    = xgb_scaler.fit_transform(X_all[train_nids_np])
X_val_tab   = xgb_scaler.transform(X_all[val_nids_np])

X_tr_xgb  = np.hstack([fused_train, X_tr_tab]).astype('float32')   # (N_tr, 327)
X_val_xgb = np.hstack([fused_val,   X_val_tab]).astype('float32')  # (N_val, 327)

print(f'XGBoost input: train={X_tr_xgb.shape}  val={X_val_xgb.shape}')
print(f'  64 (fused) + {X_tr_tab.shape[1]} (tabular)')

In [ ]:
# Train on full train set
print('
' + '='*60)
print('TRAINING FINAL XGBOOST ON FULL TRAINING SET (CPU)')
print('='*60)

start_time = time.time()
xgb_model.fit(
    X_tr_xgb, y_train_split,
    eval_set=[(X_val_xgb, y_val_split)],
    verbose=50
)
elapsed = time.time() - start_time
print(f'Final model training completed in {elapsed:.1f} seconds')

try:
    best_iter = xgb_model.best_iteration
    print(f'Best iteration: {best_iter}')
except:
    best_iter = xgb_model.n_estimators - 1
    print(f'Trained for {best_iter + 1} iterations')

## 7. Evaluation on Validation AND Test Sets

In [ ]:
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns

print("="*80)
print("EVALUATION ON VALIDATION SET AND TEST SET")
print("="*80)

# Make predictions on VALIDATION set
y_pred_proba_val = xgb_model.predict_proba(X_val_xgb)[:, 1]

# Find optimal threshold using PR curve
precisions, recalls, thresholds = precision_recall_curve(y_val_split, y_pred_proba_val)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else 0.5

y_pred_val = (y_pred_proba_val >= optimal_threshold).astype(int)

print(f"\nOptimal Threshold: {optimal_threshold:.4f}")
print(f"Optimal F1 Score: {f1_scores[optimal_idx]:.4f}")

# ============ VALIDATION SET METRICS ============
print("\n" + "="*80)
print("VALIDATION SET METRICS ({:,} rows)".format(len(y_val_split)))
print("="*80)

auc_val = roc_auc_score(y_val_split, y_pred_proba_val)
accuracy_val = accuracy_score(y_val_split, y_pred_val)
precision_val = precision_score(y_val_split, y_pred_val)
recall_val = recall_score(y_val_split, y_pred_val)
f1_val = f1_score(y_val_split, y_pred_val)

tn_val, fp_val, fn_val, tp_val = confusion_matrix(y_val_split, y_pred_val).ravel()
specificity_val = tn_val / (tn_val + fp_val) if (tn_val + fp_val) > 0 else 0

print(f"AUC-ROC:          {auc_val:.4f}")
print(f"Accuracy:         {accuracy_val:.4f}")
print(f"Precision:        {precision_val:.4f}")
print(f"Recall:           {recall_val:.4f}")
print(f"F1-Score:         {f1_val:.4f}")
print(f"Specificity:      {specificity_val:.4f}")

print("\nConfusion Matrix (Validation):")
print(f"  TP={tp_val:,}  FP={fp_val:,}")
print(f"  FN={fn_val:,}  TN={tn_val:,}")

# ============ TEST SET METRICS ============
print("\n" + "="*80)
print("TEST SET METRICS ({:,} rows)".format(len(y_test)))
print("="*80)

# Make predictions on TEST set using same threshold
y_pred_proba_test = xgb_model.predict_proba(X_test_xgb)[:, 1]
y_pred_test = (y_pred_proba_test >= optimal_threshold).astype(int)

auc_test = roc_auc_score(y_test, y_pred_proba_test)
accuracy_test = accuracy_score(y_test, y_pred_test)
precision_test = precision_score(y_test, y_pred_test)
recall_test = recall_score(y_test, y_pred_test)
f1_test = f1_score(y_test, y_pred_test)

tn_test, fp_test, fn_test, tp_test = confusion_matrix(y_test, y_pred_test).ravel()
specificity_test = tn_test / (tn_test + fp_test) if (tn_test + fp_test) > 0 else 0

print(f"AUC-ROC:          {auc_test:.4f}")
print(f"Accuracy:         {accuracy_test:.4f}")
print(f"Precision:        {precision_test:.4f}")
print(f"Recall:           {recall_test:.4f}")
print(f"F1-Score:         {f1_test:.4f}")
print(f"Specificity:      {specificity_test:.4f}")

print("\nConfusion Matrix (Test):")
print(f"  TP={tp_test:,}  FP={fp_test:,}")
print(f"  FN={fn_test:,}  TN={tn_test:,}")

# ============ COMPARISON ============
print("\n" + "="*80)
print("VALIDATION VS TEST COMPARISON")
print("="*80)

comparison_df = pd.DataFrame({
    'Metric': ['AUC', 'Accuracy', 'Precision', 'Recall', 'F1', 'Specificity'],
    'Validation': [auc_val, accuracy_val, precision_val, recall_val, f1_val, specificity_val],
    'Test': [auc_test, accuracy_test, precision_test, recall_test, f1_test, specificity_test],
})

comparison_df['Diff (Test-Val)'] = comparison_df['Test'] - comparison_df['Validation']
comparison_df = comparison_df.round(4)

print("\n" + comparison_df.to_string(index=False))

print("\n" + "="*80)
print("DATA DISTRIBUTION")
print("="*80)
print(f"\nValidation Set:")
print(f"  Benign: {(y_val_split==0).sum():,} ({(y_val_split==0).sum()/len(y_val_split)*100:.1f}%)")
print(f"  Fraud:  {(y_val_split==1).sum():,} ({(y_val_split==1).sum()/len(y_val_split)*100:.1f}%)")

print(f"\nTest Set:")
print(f"  Benign: {(y_test==0).sum():,} ({(y_test==0).sum()/len(y_test)*100:.1f}%)")
print(f"  Fraud:  {(y_test==1).sum():,} ({(y_test==1).sum()/len(y_test)*100:.1f}%)")

# Store for later use
metrics_dict = {
    'validation': {
        'auc': float(auc_val), 'accuracy': float(accuracy_val),
        'precision': float(precision_val), 'recall': float(recall_val),
        'f1': float(f1_val), 'specificity': float(specificity_val),
    },
    'test': {
        'auc': float(auc_test), 'accuracy': float(accuracy_test),
        'precision': float(precision_test), 'recall': float(recall_test),
        'f1': float(f1_test), 'specificity': float(specificity_test),
    }
}

print("\n" + "="*80)
print("GENERALIZATION ASSESSMENT")
print("="*80)
if abs(auc_test - auc_val) < 0.01:
    print("[OK] Model generalizes well (Test AUC ~ Val AUC)")
elif auc_test < auc_val:
    gap = auc_val - auc_test
    print(f"[INFO] Slight performance drop on test set (gap: {gap:.4f})")
else:
    print("[GOOD] Model performs BETTER on test set")
print("="*80)

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_validate
import time

# Base parameters
XGB_BASE = {
    'scale_pos_weight': 27,
    'tree_method': 'hist',
    'eval_metric': 'auc',
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': 0,
}

# Hyperparameter grid for tuning
param_grid = {
    'max_depth': [5, 6, 7],
    'learning_rate': [0.03, 0.05, 0.07],
    'n_estimators': [300, 500],
    'subsample': [0.7, 0.8],
    'colsample_bytree': [0.8, 0.9],
}

print('Hyperparameter Grid:')
print(f'  max_depth: {param_grid["max_depth"]}')
print(f'  learning_rate: {param_grid["learning_rate"]}')
print(f'  n_estimators: {param_grid["n_estimators"]}')
print(f'  subsample: {param_grid["subsample"]}')
print(f'  colsample_bytree: {param_grid["colsample_bytree"]}')
print(f'Total combinations: {3*3*2*2*2} = 72 configurations')
print('Note: Using 5-fold CV on each = 360 model trainings total')
print('Estimated time: 9-10 minutes on CPU (with n_jobs=-1 parallelization)')

TUNE_HYPERPARAMS = True  # Set to False to skip and use default params

## 6c. K-Fold XGBoost with GridSearchCV

Train multiple XGBoost models with 5-fold CV and hyperparameter tuning.

In [ ]:
if TUNE_HYPERPARAMS:
    print('='*60)
    print('HYPERPARAMETER TUNING VIA GRIDSEARCHCV')
    print('='*60)
    
    # GridSearchCV with 5-fold cross-validation
    grid_search = GridSearchCV(
        xgb.XGBClassifier(**XGB_BASE),
        param_grid,
        cv=5,
        scoring='roc_auc',
        n_jobs=-1,
        verbose=2
    )
    
    print('Starting GridSearchCV (5-fold CV on 72 configurations)...')
    start_time = time.time()
    grid_search.fit(X_tr_xgb, y_train_split)
    elapsed = time.time() - start_time
    
    print(f'GridSearchCV completed in {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)')
    print(f'Best parameters: {grid_search.best_params_}')
    print(f'Best CV AUC: {grid_search.best_score_:.4f}')
    
    # Use best parameters
    best_params = {**XGB_BASE, **grid_search.best_params_}
    xgb_model = xgb.XGBClassifier(**best_params)
    
else:
    # Use default parameters
    best_params = {
        'max_depth': 6,
        'n_estimators': 500,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        **XGB_BASE
    }
    xgb_model = xgb.XGBClassifier(**best_params)
    print('Using default XGBoost parameters (skipped tuning)')

# Train on full train set
print('' + '='*60)
print('TRAINING FINAL XGBOOST ON FULL TRAINING SET')
print('='*60)

start_time = time.time()
xgb_model.fit(
    X_tr_xgb, y_train_split,
    eval_set=[(X_val_xgb, y_val_split)],
    verbose=50
)
elapsed = time.time() - start_time
print(f'Final model training completed in {elapsed:.1f} seconds')

# K-Fold CV for robustness
print('' + '='*60)
print('5-FOLD CROSS-VALIDATION (Robustness Estimate)')
print('='*60)

kfold_model = xgb.XGBClassifier(**best_params)
start_time = time.time()
kfold_scores = cross_validate(
    kfold_model, X_tr_xgb, y_train_split,
    cv=5,
    scoring=['roc_auc', 'f1', 'accuracy', 'precision', 'recall'],
    return_train_score=True,
    n_jobs=-1
)
elapsed = time.time() - start_time
print(f'K-Fold CV completed in {elapsed:.1f} seconds')

print('K-Fold Results (5 folds):')
for metric in ['roc_auc', 'f1', 'accuracy', 'precision', 'recall']:
    train_m = kfold_scores[f'train_{metric}']
    test_m = kfold_scores[f'test_{metric}']
    print(f'  {metric:10s}: Train {train_m.mean():.4f}+/-{train_m.std():.4f}  | Val {test_m.mean():.4f}+/-{test_m.std():.4f}')

xgb_model.save_model(os.path.join(SAVE_DIR, 'models', 'xgb_fusion_tuned.json'))
print('Final model saved.')

## 7. Comprehensive Evaluation Metrics

Calculate all metrics: AUC, Accuracy, Precision, Recall, F1, Confusion Matrix

In [ ]:
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

# Predictions
y_proba = xgb_model.predict_proba(X_val_xgb)[:, 1]
auc_val = roc_auc_score(y_val_split, y_proba)

# Optimal threshold
precisions, recalls, thresholds = precision_recall_curve(y_val_split, y_proba)
f1s = 2*precisions*recalls/(precisions+recalls+1e-8)
best_idx = np.argmax(f1s)
best_thresh = thresholds[best_idx]
y_pred = (y_proba >= best_thresh).astype(int)

# Metrics
acc = accuracy_score(y_val_split, y_pred)
prec = precision_score(y_val_split, y_pred)
rec = recall_score(y_val_split, y_pred)
f1 = f1_score(y_val_split, y_pred)
cm = confusion_matrix(y_val_split, y_pred)
tn, fp, fn, tp = cm.ravel()
specificity = tn/(tn+fp)

print('='*70)
print('XGBOOST META-LEARNER -- COMPREHENSIVE EVALUATION')
print('='*70)
print(f'Validation Size: {len(y_val_split):,} samples')
print(f'Fraud: {y_val_split.sum():,} ({y_val_split.mean()*100:.2f}%) | Benign: {(1-y_val_split).sum():,}')

print(f'\\n{"METRIC":<20} {"VALUE":<10} {"DESCRIPTION":<40}')
print('-'*70)
print(f'{"AUC-ROC":<20} {auc_val:.4f}     Area Under ROC Curve')
print(f'{"Accuracy":<20} {acc:.4f}     (TP+TN)/All')
print(f'{"Precision":<20} {prec:.4f}     TP/(TP+FP) -- fraud detection accuracy')
print(f'{"Recall":<20} {rec:.4f}     TP/(TP+FN) -- catch rate')
print(f'{"F1 Score":<20} {f1:.4f}     Harmonic mean')
print(f'{"Specificity":<20} {specificity:.4f}     TN/(TN+FP) -- benign accuracy')
print(f'{"Optimal Threshold":<20} {best_thresh:.3f}      Maximizes F1')

print(f'\\nCONFUSION MATRIX:')
print(f'               Predicted Benign  Predicted Fraud')
print(f'Actual Benign: {tn:>6,}            {fp:>6,}')
print(f'Actual Fraud:  {fn:>6,}            {tp:>6,}')

print(f'\\nCLASSIFICATION REPORT:')
print(classification_report(y_val_split, y_pred, target_names=['Benign', 'Fraud']))
print('='*70)

## 8. Evaluation Plots

ROC, PR curve, confusion matrix, metrics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ROC Curve
fpr, tpr, _ = roc_curve(y_val_split, y_proba)
roc_auc = auc(fpr, tpr)
axes[0, 0].plot(fpr, tpr, color='steelblue', lw=2, label=f'XGBoost (AUC={roc_auc:.4f})')
axes[0, 0].plot([0, 1], [0, 1], color='red', lw=2, linestyle='--', label='Random')
axes[0, 0].fill_between(fpr, tpr, alpha=0.2, color='steelblue')
axes[0, 0].set_xlabel('FPR', fontsize=12); axes[0, 0].set_ylabel('TPR', fontsize=12)
axes[0, 0].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

# Precision-Recall Curve
axes[0, 1].plot(recalls, precisions, color='orange', lw=2)
axes[0, 1].fill_between(recalls, precisions, alpha=0.2, color='orange')
axes[0, 1].set_xlabel('Recall', fontsize=12); axes[0, 1].set_ylabel('Precision', fontsize=12)
axes[0, 1].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
axes[0, 1].grid(alpha=0.3)

# Confusion Matrix
import seaborn as sns
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': 'Count'}, ax=axes[1, 0])
axes[1, 0].set_xlabel('Predicted', fontsize=12); axes[1, 0].set_ylabel('Actual', fontsize=12)
axes[1, 0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
axes[1, 0].set_xticklabels(['Benign', 'Fraud']); axes[1, 0].set_yticklabels(['Benign', 'Fraud'])

# Metrics Bar
metrics = {'AUC': auc_val, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'Specificity': specificity}
colors = ['steelblue', 'orange', 'green', 'red', 'purple', 'brown']
axes[1, 1].bar(range(len(metrics)), list(metrics.values()), color=colors, alpha=0.8)
axes[1, 1].set_xticks(range(len(metrics)))
axes[1, 1].set_xticklabels(list(metrics.keys()), rotation=45)
axes[1, 1].set_ylabel('Score', fontsize=12)
axes[1, 1].set_title('All Metrics', fontsize=14, fontweight='bold')
axes[1, 1].set_ylim([0.8, 1.0]); axes[1, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(metrics.values()):
    axes[1, 1].text(i, v+0.005, f'{v:.4f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'xgb_evaluation_plots.png'), dpi=100, bbox_inches='tight')
plt.show()
print('Plots saved.')

## 9. Model Comparison: Individual vs Fusion

In [ ]:
print('='*80)
print('MODEL COMPARISON: INDIVIDUAL MODELS vs FUSION')
print('='*80)

# Compare all models
comparison = {}
for name, probs in [('TCN', gate_probs if 'gate_probs' in locals() else np.zeros(len(y_val_split))),
                      ('GAT', np.zeros(len(y_val_split))),
                      ('CaT-GNN', np.zeros(len(y_val_split))),
                      ('Gated Fusion', np.zeros(len(y_val_split))),
                      ('XGBoost', y_proba)]:
    if name == 'XGBoost':
        a = auc_val
        pred = y_pred
        acc_m = acc
        prec_m = prec
        rec_m = rec
        f1_m = f1
    else:
        continue
    comparison[name] = {'AUC': a, 'Acc': acc_m, 'Prec': prec_m, 'Rec': rec_m, 'F1': f1_m}

print(f'\\n{"Model":<25} {"AUC":<8} {"Accuracy":<10} {"Precision":<10} {"Recall":<8} {"F1":<8}')
print('-'*70)
for model, metrics in comparison.items():
    print(f'{model:<25} {metrics["AUC"]:.4f}   {metrics["Acc"]:.4f}     {metrics["Prec"]:.4f}      {metrics["Rec"]:.4f}   {metrics["F1"]:.4f}')

print('='*80)

In [ ]:
# Save evaluation results
eval_results = {
    'xgb_metrics': {
        'auc': float(auc_val), 'accuracy': float(acc), 'precision': float(prec),
        'recall': float(rec), 'f1': float(f1), 'specificity': float(specificity),
        'threshold': float(best_thresh),
        'cm': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)},
    },
    'kfold_cv': {
        'auc': (float(kfold_scores['test_roc_auc'].mean()), float(kfold_scores['test_roc_auc'].std())),
        'f1': (float(kfold_scores['test_f1'].mean()), float(kfold_scores['test_f1'].std())),
        'accuracy': (float(kfold_scores['test_accuracy'].mean()), float(kfold_scores['test_accuracy'].std())),
    },
    'hyperparameters': best_params,
}

with open(os.path.join(SAVE_DIR, 'fusion_evaluation_comprehensive.json'), 'w') as f:
    json.dump(eval_results, f, indent=2)

print('All results saved to fusion_results/')

In [ ]:
y_proba = xgb_model.predict_proba(X_val_xgb)[:,1]
auc     = roc_auc_score(y_val_split, y_proba)

precisions, recalls, thresholds = precision_recall_curve(y_val_split, y_proba)
f1s         = 2*precisions*recalls/(precisions+recalls+1e-8)
best_idx    = np.argmax(f1s)
best_thresh = thresholds[best_idx]
y_pred      = (y_proba >= best_thresh).astype(int)
f1          = f1_score(y_val_split, y_pred)

# Neural gate AUC alone
fusion_clf.eval()
fusion_clf.to(DEVICE)
with torch.no_grad():
    vl, _ = fusion_clf(t_tcn_val.to(DEVICE), t_gat_val.to(DEVICE), t_cat_val.to(DEVICE))
    gate_probs = torch.sigmoid(vl.squeeze(-1)).cpu().numpy()
gate_auc = roc_auc_score(y_val_split, gate_probs)

print('='*55)
print('FUSION RESULTS')
print('='*55)
print(f'Gated Fusion AUC (neural):  {gate_auc:.4f}')
print(f'XGBoost Meta-Learner AUC:   {auc:.4f}')
print(f'XGBoost F1:                 {f1:.4f}')
print(f'Optimal threshold:          {best_thresh:.3f}')
print(f'XGBoost improvement:        +{(auc-gate_auc)*100:.2f} pp')
print('='*55)

In [ ]:
fraud_v  = y_val_split==1
benign_v = y_val_split==0

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(['TCN','GAT','CaT-GNN'],
            [gw_val[fraud_v,i].mean() for i in range(3)],
            color=['steelblue','orange','green'], alpha=0.8, label='Fraud')
axes[0].bar(['TCN','GAT','CaT-GNN'],
            [gw_val[benign_v,i].mean() for i in range(3)],
            color=['steelblue','orange','green'], alpha=0.4, label='Benign')
axes[0].set_title('Gate Weights: Fraud vs Benign (Validation)')
axes[0].set_ylabel('Mean Gate Weight'); axes[0].legend()

feat_names = [f'fused_{i}' for i in range(64)] + list(cols)
n_fi = len(xgb_model.feature_importances_)
importance = pd.Series(xgb_model.feature_importances_, index=feat_names[:n_fi])
importance.nlargest(20).sort_values().plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top-20 XGBoost Feature Importance')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'fusion_analysis.png'), dpi=100, bbox_inches='tight')
plt.show()

print('Gate weights (validation):')
print(f'  Fraud:  TCN={gw_val[fraud_v,0].mean():.3f} GAT={gw_val[fraud_v,1].mean():.3f} CaT={gw_val[fraud_v,2].mean():.3f}')
print(f'  Benign: TCN={gw_val[benign_v,0].mean():.3f} GAT={gw_val[benign_v,1].mean():.3f} CaT={gw_val[benign_v,2].mean():.3f}')

In [ ]:
results = {
    'xgb_auc':       float(auc),
    'xgb_f1':        float(f1),
    'xgb_threshold': float(best_thresh),
    'gate_auc':      float(gate_auc),
    'gate_weights_avg': {
        'TCN':     float(gw_val[:,0].mean()),
        'GAT':     float(gw_val[:,1].mean()),
        'CaT-GNN': float(gw_val[:,2].mean())
    },
    'fusion_training_history': history,
}
with open(os.path.join(SAVE_DIR,'fusion_metrics.json'),'w') as f:
    json.dump(results, f, indent=2)

with open(os.path.join(SAVE_DIR,'models','xgb_scaler.pkl'),'wb') as f:
    pickle.dump(xgb_scaler, f)

print('Saved to fusion_results/')
print(f'XGBoost AUC: {auc:.4f} | F1: {f1:.4f} | Threshold: {best_thresh:.3f}')

## Test Set: Complete Embedding Extraction & Evaluation

Full pipeline aligned with train/val preprocessing.

In [ ]:
# ── A. Prepare test feature matrix (IDENTICAL to Cell 12 for train) ─────────
print('='*70)
print('TEST FEATURE MATRIX PREPARATION')
print('='*70)

# Exact same logic as Cell 12:
#   X_all = X_train[cols].copy().fillna(-1).astype('float32').values
X_test_all = X_test[cols].copy().fillna(-1).astype('float32').values

print(f'X_test_all shape: {X_test_all.shape}')
print(f'X_all (train) shape: {X_all.shape}')
print(f'Match: {X_test_all.shape[1] == X_all.shape[1]}')

assert X_test_all.shape[1] == X_all.shape[1], \
    f'Feature mismatch: test={X_test_all.shape[1]}, train={X_all.shape[1]}'

# Also build test relation columns needed for graph (same as Cell 10 saved for train)
uid_test    = X_test['uid'].copy()
card1_test  = X_test['card1'].copy()
pemail_test = X_test['P_emaildomain'].copy()
device_test = X_test['DeviceInfo'].copy()

print(f'\nTest relation columns saved for graph construction.')
print('[OK] Test feature matrix ready')

In [ ]:
# ── B. Build combined graph (train + test) for GAT / CaT-GNN ────────────────
# The GTAN graph in Cell 14 only contained train nodes.
# For test extraction we need test nodes in the graph too.
print('='*70)
print('BUILDING COMBINED GRAPH (TRAIN + TEST)')
print('='*70)

n_train = len(X_all)
n_test  = len(X_test_all)
n_total = n_train + n_test

# Combine feature matrix: train nodes first, then test nodes
X_combined = np.vstack([X_all, X_test_all])   # (n_total, IN_DIM)
print(f'Train nodes:    {n_train:,}')
print(f'Test nodes:     {n_test:,}')
print(f'Combined nodes: {n_total:,}')

# Scale combined features (fit only on train nodes, same as Cell 13)
graph_scaler_comb = StandardScaler()
X_combined_scaled = X_combined.copy()
X_combined_scaled[:n_train] = graph_scaler_comb.fit_transform(X_combined[:n_train])
X_combined_scaled[n_train:] = graph_scaler_comb.transform(X_combined[n_train:])

# Combined node features tensor
node_feat_comb = torch.FloatTensor(X_combined_scaled)

# Combined relation series: shift test indices by n_train
uid_comb    = pd.concat([uid_train.reset_index(drop=True),
                         uid_test.reset_index(drop=True).rename(lambda x: x+n_train)],
                        axis=0)
card1_comb  = pd.concat([card1_train.reset_index(drop=True),
                         card1_test.reset_index(drop=True).rename(lambda x: x+n_train)],
                        axis=0)
pemail_comb = pd.concat([pemail_train.reset_index(drop=True),
                         pemail_test.reset_index(drop=True).rename(lambda x: x+n_train)],
                        axis=0)
device_comb = pd.concat([device_train.reset_index(drop=True),
                         device_test.reset_index(drop=True).rename(lambda x: x+n_train)],
                        axis=0)

# Build GTAN graph with combined transactions
print('\nBuilding GTAN graph...')
t0 = time.time()
g_comb, _ = build_gtan_graph(
    n_nodes=n_total,
    relation_series_dict={
        'uid':           uid_comb,
        'card1':         card1_comb,
        'P_emaildomain': pemail_comb,
        'DeviceInfo':    device_comb,
    },
    k=GRAPH_K_NEIGHBORS, max_group=MAX_GROUP_SIZE
)
g_comb.ndata['feat'] = node_feat_comb
print(f'Combined graph built in {time.time()-t0:.1f}s')

# Test node IDs in combined graph are indices [n_train, n_total)
test_nids_np   = np.arange(n_train, n_total)
test_nids_tensor = torch.LongTensor(test_nids_np)

print(f'\nTest node IDs: [{n_train:,}, {n_total:,})')
print(f'Test node count: {len(test_nids_np):,}')
print('[OK] Combined graph ready for embedding extraction')

In [ ]:
# ── C. TCN Embedding Extraction - Test Set ────────────────────────────────────
print('='*70)
print('TCN EMBEDDING EXTRACTION - TEST SET')
print('='*70)

# Use tcn_scaler (fitted on train split in experiment_tcn / saved as pkl)
X_test_scaled_tcn = tcn_scaler.transform(X_test_all)          # same scaler

n_pad = 280 - X_test_all.shape[1]                             # same padding
X_test_padded = np.hstack([X_test_scaled_tcn,
                            np.zeros((len(X_test_scaled_tcn), n_pad))])
X_test_tcn = X_test_padded.reshape(-1, 20, 14).astype('float32')  # same reshape

tcn_emb_test = tcn_embedder.predict(X_test_tcn, batch_size=1024)   # same batch_size

print(f'TCN test embeddings: {tcn_emb_test.shape}')
assert tcn_emb_test.shape == (len(X_test_all), 64), 'Shape mismatch'
print('[OK] TCN test embeddings extracted')

In [ ]:
# ── D. GAT Embedding Extraction - Test Set ───────────────────────────────────
print('='*70)
print('GAT EMBEDDING EXTRACTION - TEST SET')
print('='*70)

sampler_gat_t = MultiLayerFullNeighborSampler(2)              # same sampler depth
loader_gat_t  = dgl.dataloading.DataLoader(
    g_comb, test_nids_tensor, sampler_gat_t,
    batch_size=4096, shuffle=False, drop_last=False            # same batch_size
)

gat_model.eval()
gat_model.to(DEVICE)

gat_emb_list_t = []
with torch.no_grad():
    for input_nodes, output_nodes, blocks in loader_gat_t:
        x = node_feat_comb[input_nodes].to(DEVICE)
        blocks = [b.to(DEVICE) for b in blocks]
        h = gat_model(blocks, x)
        gat_emb_list_t.append(h.cpu().numpy())

gat_emb_test = np.vstack(gat_emb_list_t)

print(f'GAT test embeddings: {gat_emb_test.shape}')
assert gat_emb_test.shape == (len(test_nids_np), 64), 'Shape mismatch'
print('[OK] GAT test embeddings extracted')

In [ ]:
# ── E. CaT-GNN Embedding Extraction - Test Set ───────────────────────────────
print('='*70)
print('CaT-GNN EMBEDDING EXTRACTION - TEST SET')
print('='*70)

sampler_cat_t = MultiLayerFullNeighborSampler(1)              # same sampler depth
loader_cat_t  = dgl.dataloading.DataLoader(
    g_comb, test_nids_tensor, sampler_cat_t,
    batch_size=2048, shuffle=False, drop_last=False            # same batch_size
)

catgnn_model.eval()
catgnn_model.to(DEVICE)

cat_emb_list_t = []
with torch.no_grad():
    for input_nodes, output_nodes, blocks in loader_cat_t:
        x = node_feat_comb[input_nodes].to(DEVICE)
        blocks = [b.to(DEVICE) for b in blocks]
        h = catgnn_model(blocks[0], x)
        cat_emb_list_t.append(h.cpu().numpy())

catgnn_emb_test = np.vstack(cat_emb_list_t)

print(f'CaT-GNN test embeddings: {catgnn_emb_test.shape}')
assert catgnn_emb_test.shape == (len(test_nids_np), 64), 'Shape mismatch'
print('[OK] CaT-GNN test embeddings extracted')

In [ ]:
# ── F. Gated Fusion - Test Set ───────────────────────────────────────────────
print('='*70)
print('GATED FUSION - TEST SET')
print('='*70)

t_tcn_test = torch.FloatTensor(tcn_emb_test)
t_gat_test = torch.FloatTensor(gat_emb_test)
t_cat_test = torch.FloatTensor(catgnn_emb_test)

fusion_clf.eval()
fusion_clf.to(DEVICE)

with torch.no_grad():
    fused_test, gate_w_test = fusion_clf.fusion(
        t_tcn_test.to(DEVICE),
        t_gat_test.to(DEVICE),
        t_cat_test.to(DEVICE)
    )

fused_test   = fused_test.cpu().numpy()
gate_w_test  = gate_w_test.cpu().numpy()

print(f'Fused test shape: {fused_test.shape}')
avg = gate_w_test.mean(axis=0)
print(f'Mean gate weights - TCN:{avg[0]:.3f}  GAT:{avg[1]:.3f}  CaT:{avg[2]:.3f}')
assert np.allclose(gate_w_test.sum(axis=1), 1.0, atol=1e-5), 'Gate weights must sum to 1'
print('[OK] Gated fusion complete')

In [ ]:
# ── G. XGBoost Predictions - Test Set ────────────────────────────────────────
print('='*70)
print('XGBOOST PREDICTIONS - TEST SET')
print('='*70)

# Scale tabular features using SAME xgb_scaler fitted on train split
X_test_tab  = xgb_scaler.transform(X_test_all)                # same scaler
X_test_xgb  = np.hstack([fused_test, X_test_tab])             # same 64+263=327

print(f'X_test_xgb shape: {X_test_xgb.shape}')
assert X_test_xgb.shape[1] == 327, 'Expected 327 features'

y_test_proba = xgb_model.predict_proba(X_test_xgb)[:, 1]     # same model
y_test_pred  = (y_test_proba >= optimal_threshold).astype(int) # same threshold

print(f'Predictions: {len(y_test_proba):,} samples')
print(f'Threshold:   {optimal_threshold:.4f}')
print(f'Benign (0):  {(y_test_pred==0).sum():,}  ({(y_test_pred==0).mean()*100:.2f}%)')
print(f'Fraud  (1):  {(y_test_pred==1).sum():,}  ({(y_test_pred==1).mean()*100:.2f}%)')
print('[OK] XGBoost predictions complete')

In [ ]:
# ── H. Test Set Evaluation & Comparison with Validation ─────────────────────
print('='*70)
print('TEST SET EVALUATION & COMPARISON WITH VALIDATION')
print('='*70)

# y_test labels: rows of y_all NOT in train_nids_np / val_nids_np
# Since X_all only had train data, test labels come from y_test (if available)
try:
    y_test_labels = y_test.values if hasattr(y_test, 'values') else np.array(y_test)

    auc_test      = roc_auc_score(y_test_labels, y_test_proba)
    acc_test      = accuracy_score(y_test_labels, y_test_pred)
    prec_test     = precision_score(y_test_labels, y_test_pred)
    rec_test      = recall_score(y_test_labels, y_test_pred)
    f1_test       = f1_score(y_test_labels, y_test_pred)
    tn_t,fp_t,fn_t,tp_t = confusion_matrix(y_test_labels, y_test_pred).ravel()
    spec_test     = tn_t / (tn_t + fp_t)

    print('\nTest Set Metrics:')
    print(f'  AUC-ROC:     {auc_test:.4f}')
    print(f'  Accuracy:    {acc_test:.4f}')
    print(f'  Precision:   {prec_test:.4f}')
    print(f'  Recall:      {rec_test:.4f}')
    print(f'  F1-Score:    {f1_test:.4f}')
    print(f'  Specificity: {spec_test:.4f}')

    print('\nConfusion Matrix (Test):')
    print(f'  TN={tn_t:,}  FP={fp_t:,}')
    print(f'  FN={fn_t:,}  TP={tp_t:,}')

    print('\nClassification Report (Test):')
    print(classification_report(y_test_labels, y_test_pred,
                                target_names=['Benign','Fraud']))

    print('\n' + '='*70)
    print('TEST vs VALIDATION COMPARISON')
    print('='*70)
    print(f'{"Metric":<15} {"Validation":>12} {"Test":>12} {"Diff":>10}')
    print('-'*52)
    for metric, v_val, t_val in [
        ('AUC-ROC',    auc_full,         auc_test),
        ('Accuracy',   accuracy_full,    acc_test),
        ('Precision',  precision_full,   prec_test),
        ('Recall',     recall_full,      rec_test),
        ('F1-Score',   f1_full,          f1_test),
        ('Specificity',specificity_full, spec_test),
    ]:
        diff = t_val - v_val
        print(f'{metric:<15} {v_val:>12.4f} {t_val:>12.4f} {diff:>+10.4f}')
    print('-'*52)

    auc_gap = abs(auc_test - auc_full)
    print(f'\nGeneralization gap (AUC): {auc_gap:.4f}', end='  ')
    if auc_gap < 0.01:   print('[EXCELLENT]')
    elif auc_gap < 0.02: print('[GOOD]')
    elif auc_gap < 0.05: print('[OK]')
    else:                print('[WARNING - large gap]')

    # Save
    import json as _json
    comparison = {
        'validation': {'auc': float(auc_full), 'accuracy': float(accuracy_full),
                       'precision': float(precision_full), 'recall': float(recall_full),
                       'f1': float(f1_full), 'specificity': float(specificity_full)},
        'test':       {'auc': float(auc_test), 'accuracy': float(acc_test),
                       'precision': float(prec_test), 'recall': float(rec_test),
                       'f1': float(f1_test), 'specificity': float(spec_test)},
        'auc_gap': float(auc_gap),
    }
    with open(os.path.join(SAVE_DIR, 'test_vs_val_comparison.json'), 'w') as _f:
        _json.dump(comparison, _f, indent=2)
    print(f'\nResults saved to: {SAVE_DIR}/test_vs_val_comparison.json')

except Exception as e:
    print(f'\n[INFO] Test labels not available: {str(e)[:120]}')
    print('Saving submission predictions...')
    sub = pd.DataFrame({'TransactionID': X_test.index,
                        'isFraud': y_test_proba})
    sub.to_csv(os.path.join(SAVE_DIR, 'submission.csv'), index=False)
    print(f'Submission saved to: {SAVE_DIR}/submission.csv')
    print(f'Predictions: {len(y_test_proba):,} rows')